# PATH MANAGEMENT

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

%load_ext autoreload
%autoreload 2
# %matplotlib inline

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/notebooks
/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/app


In [2]:
from src.config import Configuration

CONFIG = Configuration(
    batch_size=32,
    max_tok_length=16,

    num_shots=5,
)

# Prompting

Prompting in LLMs is the design of a structured input to provide task description, demostrations and the actual input for the model to generate a desired output.

In this notebook, we are going to use for fine-tuning a dataset set that is already available in the [Datasets repository](https://huggingface.co/datasets) from Hugging Face. However, the [Datasets library](https://huggingface.co/docs/datasets) makes easy to access and load datasets. For example, you can easily load your own dataset following [this tutorial](https://huggingface.co/docs/datasets/loading#local-and-remote-files).

More precisely, we are going to explain how to perform In-Context Learning with the [Llama2 model](https://huggingface.co/docs/transformers/model_doc/llama2) on the [Europarl-ST dataset](https://huggingface.co/datasets/tj-solergibert/Europarl-ST), but only that [dataset of Europarl-ST focused on the text data for MT from English](https://huggingface.co/datasets/tj-solergibert/Europarl-ST-processed-mt-en).

In [3]:
# from datasets import load_dataset

# raw_datasets = load_dataset("tj-solergibert/Europarl-ST-processed-mt-en")

# print(raw_datasets)

from src.data import get_es_eo_dataset

raw_datasets = get_es_eo_dataset(CONFIG)

print(raw_datasets)

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 200965
    })
    test: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 43063
    })
    valid: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 43063
    })
})


As shown, the Europarl-ST already comes with a pre-defined partition on the three conventional sets: training, validation and test. Each set is a dictionary with a list of source sentences (source_text), target sentences (dest_text) and the target language (dest_lang).

Let's take a closer look at the features of the training set:

In [4]:
raw_datasets["train"].features

{'source_text': Value('string'),
 'dest_text': Value('string'),
 'dest_lang': Value('int64')}

As you can see, the possible target languages are German, English, Spanish, French, Italian, Dutch, Polish, Portuguese and Romanian.

Let us take a look at the translations of the first two English sentences:

In [5]:
raw_datasets["train"][:14]["source_text"]

['artículo anteriorse devela un misterio: ¿por qué moni argento era de tostado?',
 'en el siglo iii surgieron un número de tribus germánicas del oeste grandes: alemanni, francos, catos, \ufeffsajones, frisii, \ufeffsicambri, y thuringii\ufeff.',
 'hubo unos 200 invitados.',
 '¿eres tú mayor de edad?',
 '-"pero no tienes dinero, ¿verdad?"',
 'así que, cuando ella te deja, ¿de donde crees que ella va a hacer a continuación.',
 'para empezar, como ya hemos dicho, debemos tomar la fruta con el estómago vacío.',
 'soy una viuda con cuatro hijos y me quedé atrapado en una situación financiera desde abril de 2016 y necesitaba refinanciar y pagar mis cuentas.',
 'el trabajo con espacios en blanco debe comenzar a fines de la primavera o principios del verano y no retrasarse hasta el otoño para evitar problemas e interrupciones.',
 'es la riqueza guardada por su dueño para su propia desgracia.',
 'buscamos una canción que trate sobre alguno de los siguientes temas: «desarrollo global» o «un solo

In [6]:
raw_datasets["train"][:14]["dest_text"]

['estis mistero por la polico : kial ŝteli nur unu ŝuon anstataù paro ?',
 'la tria jarcento vidis la aperon de kelkaj grandaj okcident ĝermanaj triboj: la alemanoj, frankoj, bavarii-, ĥatoj, saksoj, frisii, sicambri, kaj thuringii.',
 'venis ĉirkaŭ 200 gastoj.',
 'ĉu vi estas la plej aĝa?',
 '"sed vi ne posedas tiom da mono, ĉu ne?"',
 'do, kiam ŝi lasas vin, kie vi kredas, ke ŝi faros poste.',
 'kiel antaŭe menciite, la drogo devas esti prenita sur malplena stomako.',
 'en ĉi tiu tempo mi estas vidvino kun kvar infanoj kaj mi estis ligita en financa situacio en majo 2018 kaj bezonis refinanci kaj pagi miajn biletojn.',
 'laboro kun spacoj devas komenciĝi fine de printempo aŭ frua somero kaj ne malhelpu ĝis aŭtuno por eviti problemojn kaj interrompojn.',
 'riĉecon konservatan por la malutilo de ĝia propra mastro.',
 'tie ĉi mi menciu nur unu temaron, tiun de tutmondiĝo aŭ „globaliĝo”.',
 'ni serĉu rekte la titolon «orientaj tapiŝoj»!',
 'tamen, ĉu tranĉeoj estas por ke ni koncentriĝu'

In [7]:
raw_datasets["train"][:14]["dest_lang"]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

As shown, each English sentence is repeated for each of the seven target languages (0: 'de', 2: 'es', 3: 'fr', 4: 'it', 5: 'nl', 6: 'pl', 7: 'pt').

The Llama2 model is a pretrained Large Language Model (LLM) ready to tackle several NLP tasks, being one of the them the translation from English into Spanish. Let us filter the Europarl-ST only for English into Spanish using a simple [lambda function](https://realpython.com/python-lambda/) with the [Dataset.filter() function](https://huggingface.co/docs/datasets/v2.9.0/en/package_reference/main_classes#datasets.Dataset.filter).

In [8]:
# lang="es"
# lang_id = raw_datasets["train"].features["dest_lang"].names.index(lang)
# raw_datasets = raw_datasets.filter(lambda x: x["dest_lang"] == lang_id)

More precisely, we are going to be using the Llama-2 checkpoint [meta-llama/Llama-2-7b-hf](https://huggingface.co/meta-llama/Llama-2-7b-hf) to run our experiments for which you need to accept the LLAMA 2 COMMUNITY LICENSE AGREEMENT. Processing your request may take some time, so please do it in advance.

Logging in HuggingFace to be granted access to Llama2 with 7B parameters:

In [9]:
import os
import dotenv
from huggingface_hub import login

dotenv.load_dotenv()
login(token=os.getenv("HUGGING_FACE_TOKEN"))

python-dotenv could not parse statement starting at line 2


python-dotenv could not parse statement starting at line 4


python-dotenv could not parse statement starting at line 5


python-dotenv could not parse statement starting at line 7


python-dotenv could not parse statement starting at line 8


We can apply the tokenizer function to any dataset taking advantage that Hugging Face Datasets are [Apache Arrow](https://arrow.apache.org) files stored on the disk, so you only keep the samples you ask for loaded in memory.

To keep the data as a dataset, we will use the [Dataset.map() function](https://huggingface.co/docs/datasets/en/package_reference/main_classes#datasets.Dataset.map). This also allows us some extra flexibility, if we need more preprocessing done than just tokenization. The map() method works by applying a function on each element of the dataset.

In our case, each sample pair is going to be preprocessed according to the needs of the model that is to be prompted. In the case of Llama2, it is recommended to explicitly state a task prompt for each source sentence:

In [10]:
from transformers import AutoTokenizer

checkpoint = "meta-llama/Llama-2-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint,
    token=True,
    padding=True,
    pad_to_multiple_of=8,
    truncation=True,
    max_length=CONFIG.max_tok_length,
    padding_side='left',
    )
tokenizer.pad_token = tokenizer.eos_token

In [11]:
def preprocess_function(sample):
    model_inputs = tokenizer(
        sample["source_text"], 
        text_target = sample["dest_text"],
        )
    return model_inputs

The way the Datasets library applies this processing is by adding new fields to the datasets, one for each key in the dictionary returned by the tokenize function, that is, *input_ids*, *attention_mask* and *labels*. We can check what the preprocess_function is doing with a small sample

In [12]:
sample = raw_datasets["train"].select(range(2))
model_input = preprocess_function({
    "source_text": list(sample["source_text"]),
    "dest_text": list(sample["dest_text"]),
})
print(model_input)

{'input_ids': [[1, 1616, 21825, 14123, 344, 316, 955, 29874, 443, 286, 1531, 601, 29901, 18613, 1971, 439, 29948, 1601, 29875, 1852, 9239, 3152, 316, 304, 303, 912, 29973], [1, 427, 560, 14521, 474, 2236, 25300, 10243, 443, 13831, 316, 9434, 375, 22593, 1715, 5070, 628, 288, 4196, 13830, 29901, 20712, 9889, 29892, 2524, 3944, 29892, 274, 4507, 29892, 29871, 30143, 29879, 1175, 2873, 29892, 1424, 275, 2236, 29892, 29871, 30143, 29879, 293, 1117, 374, 29892, 343, 266, 3864, 2236, 30143, 29889]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'labels': [[1, 707, 275, 286, 1531, 29877, 1277, 425, 1248, 1417, 584, 413, 616, 29871, 31805, 29873, 5037, 5595, 443, 29884, 29871, 31805, 29884, 265, 385, 303, 532, 30071, 610, 29877, 1577], [1, 425, 260, 2849, 14631, 1760, 29877, 7840

In [13]:
for sample in model_input['input_ids']:
    print(tokenizer.convert_ids_to_tokens(sample))

['<s>', '▁art', 'ículo', '▁anterior', 'se', '▁de', 'vel', 'a', '▁un', '▁m', 'ister', 'io', ':', '▁¿', 'por', '▁qu', 'é', '▁mon', 'i', '▁arg', 'ento', '▁era', '▁de', '▁to', 'st', 'ado', '?']
['<s>', '▁en', '▁el', '▁siglo', '▁i', 'ii', '▁surg', 'ieron', '▁un', '▁número', '▁de', '▁trib', 'us', '▁germ', 'án', 'icas', '▁del', '▁o', 'este', '▁grandes', ':', '▁alem', 'anni', ',', '▁fran', 'cos', ',', '▁c', 'atos', ',', '▁', '\ufeff', 's', 'aj', 'ones', ',', '▁fr', 'is', 'ii', ',', '▁', '\ufeff', 's', 'ic', 'amb', 'ri', ',', '▁y', '▁th', 'uring', 'ii', '\ufeff', '.']


We can recover the source text by applying [batch_decode](https://huggingface.co/docs/transformers/en/internal/tokenization_utils#transformers.PreTrainedTokenizerBase.batch_decode) of the tokenizer 

In [14]:
tokenizer.batch_decode(model_input['input_ids'])

['<s> artículo anteriorse devela un misterio: ¿por qué moni argento era de tostado?',
 '<s> en el siglo iii surgieron un número de tribus germánicas del oeste grandes: alemanni, francos, catos, \ufeffsajones, frisii, \ufeffsicambri, y thuringii\ufeff.']

Now, we can apply the preprocess_function to the raw datasets (training, validation and test):

In [15]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Map:   0%|                                                        | 0/200965 [00:00<?, ? examples/s]

Map:   2%|▊                                         | 4000/200965 [00:00<00:09, 21681.54 examples/s]

Map:   6%|██▍                                      | 12000/200965 [00:00<00:04, 44213.04 examples/s]

Map:   9%|███▋                                     | 18000/200965 [00:00<00:05, 35372.00 examples/s]

Map:  12%|█████                                    | 25000/200965 [00:00<00:03, 44018.23 examples/s]

...

Map:  39%|████████████████▌                         | 17000/43063 [00:00<00:00, 34983.75 examples/s]

Map:  58%|████████████████████████▍                 | 25000/43063 [00:00<00:00, 45048.52 examples/s]

Map:  79%|█████████████████████████████████▏        | 34000/43063 [00:00<00:00, 40780.51 examples/s]

Map:  98%|████████████████████████████████████████▉ | 42000/43063 [00:00<00:00, 48273.47 examples/s]

Map: 100%|██████████████████████████████████████████| 43063/43063 [00:00<00:00, 43471.79 examples/s]

We are going to filter the tokenized datasets by maximum number of tokens in source and target language:

In [16]:
tokenized_datasets = tokenized_datasets.filter(lambda x: len(x["input_ids"]) <= CONFIG.max_tok_length and len(x["labels"]) <= CONFIG.max_tok_length , desc=f"Discarding source and target sentences with more than {CONFIG.max_tok_length} tokens")

Discarding source and target sentences with more than 16 tokens:   0%| | 0/200965 [00:00<?, ? exampl

Discarding source and target sentences with more than 16 tokens:   0%| | 1000/200965 [00:00<00:25, 7

Discarding source and target sentences with more than 16 tokens:   4%| | 9000/200965 [00:00<00:04, 4

Discarding source and target sentences with more than 16 tokens:   8%| | 17000/200965 [00:00<00:03, 

Discarding source and target sentences with more than 16 tokens:  12%| | 25000/200965 [00:00<00:03, 

...

Discarding source and target sentences with more than 16 tokens:  37%|▎| 16000/43063 [00:00<00:00, 6

Discarding source and target sentences with more than 16 tokens:  56%|▌| 24000/43063 [00:00<00:00, 6

Discarding source and target sentences with more than 16 tokens:  74%|▋| 32000/43063 [00:00<00:00, 6

Discarding source and target sentences with more than 16 tokens:  93%|▉| 40000/43063 [00:00<00:00, 6

Discarding source and target sentences with more than 16 tokens: 100%|█| 43063/43063 [00:00<00:00, 6

We can take a quick look at the length histogram in the source language:

In [17]:
dic = {}
for sample in tokenized_datasets['train']:
    sample_length = len(sample['input_ids'])
    if sample_length not in dic:
        dic[sample_length] = 1
    else:
        dic[sample_length] += 1 

for i in range(1,CONFIG.max_tok_length+1):
    if i in dic:
        print(f"{i:>2} {dic[i]:>3}")

 3  22
 4 234
 5 1075
 6 2905
 7 5647
 8 8276
 9 9821
10 9751
11 8611
12 7037
13 5188
14 3638
15 2349
16 1486


Checking a sample after filtering by maximum number of tokens:

In [18]:
for sample in tokenized_datasets['train'].select(range(5)):
    print(sample['input_ids'])
    print(sample['attention_mask'])
    print(sample['labels'])

[1, 19766, 29877, 22660, 29871, 29906, 29900, 29900, 2437, 277, 2255, 29889]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[1, 6003, 275, 29871, 31431, 381, 1335, 30520, 29871, 29906, 29900, 29900, 10489, 517, 29926, 29889]
[1, 18613, 11175, 260, 30030, 9105, 316, 1226, 328, 29973]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[1, 29871, 31431, 29884, 3516, 22388, 425, 5644, 29926, 263, 31303, 29874, 29973]
[1, 3133, 29877, 27044, 912, 1919, 25370, 1941]
[1, 1, 1, 1, 1, 1, 1, 1]
[1, 1146, 30520, 336, 5748, 29892, 3006, 912]
[1, 337, 1789, 316, 3966, 10183, 313, 29882, 5427, 29871, 29896, 29955, 29900, 29955, 29897]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[1, 2614, 417, 10112, 601, 313, 31303, 275, 29871, 29896, 29955, 29900, 29955, 29897]
[1, 1346, 29894, 14054, 17926, 443, 553, 579, 276, 5264, 8643]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[1, 6836, 22388, 6427, 440, 609, 1175, 5374, 273, 29466, 23364, 18753, 29889, 2047]


In [19]:
src = CONFIG.src_abr
tgt = CONFIG.tgt_abr
task_prefix = f"Translate from {src} to {tgt}:\n"
shots = ""
s = ""

prefix_tok_len = len(tokenizer.encode(f"{task_prefix}{shots}{src}: {s} = {tgt}: "))
shot_tok_len   = len(tokenizer.encode(f"{src}: {s} = {tgt}: {s}\n"))
max_tok_len = prefix_tok_len
max_tok_len += CONFIG.num_shots * (shot_tok_len + 2 * CONFIG.max_tok_length) 
max_tok_len += CONFIG.max_tok_length

random_seed = 13
sample = tokenized_datasets['train'].shuffle(seed=random_seed).select(range(CONFIG.num_shots))
for s in sample: shots += f"{src}: {s['source_text']} = {tgt}: {s['dest_text']}\n" 

def preprocess4test_function(sample):
    inputs = [f"{task_prefix}{shots}{src}: {s} = {tgt}: " for s in sample["source_text"]]
    model_inputs = tokenizer(
        inputs,
        max_length=max_tok_len, 
        truncation=True, 
        return_tensors="pt", 
        padding=True)
    return model_inputs

The way the Datasets library applies this processing is by adding new fields to the datasets, one for each key in the dictionary returned by the tokenize function, that is, *input_ids*, *attention_mask* and *labels*:

In [20]:
sample = tokenized_datasets['test'].select(range(5))
model_input = preprocess4test_function(sample)
print(model_input)
print(tokenizer.batch_decode(model_input['input_ids']))

{'input_ids': tensor([[    1,  4103,  9632,   515,   831,   304,   321, 29877, 29901,    13,
           267, 29901, 26692,   831,  1321,  8154,  2407,   279,  5409, 29889,
           353,   321, 29877, 29901, 22388, 29871, 31431,  2829,   288,   637,
          1540,   337,  1397, 29875,   980,   262, 29889,    13,   267, 29901,
          1869,  7014,   316, 17176,   328,   343,   353,   321, 29877, 29901,
           425,  1957,  3848,  3691,  4931,   423,  4303,   406,  1111,   413,
          1175,    13,   267, 29901,  5516,   425,  3638, 29874,   316,  1232,
          4439,   359,   282,   406, 23322, 29889,   353,   321, 29877, 29901,
          7048,   425,   992,  2212,   316,  4439,  1631,   352,  3848,   282,
           406,   359, 29889,    13,   267, 29901,  1354,  1750,   406, 21415,
         20397, 17639,   353,   321, 29877, 29901,   409,  3516,  1700,   294,
          2071,  1091, 29875,  1375,    13,   267, 29901,   831,  3079, 29983,
           569,  2689,  6062,  4778, 2

In [21]:
preprocessed_test_dataset = tokenized_datasets['test'].map(preprocess4test_function, batched=True)

Map:   0%|                                                         | 0/14342 [00:00<?, ? examples/s]

Map:  14%|█████▉                                     | 2000/14342 [00:00<00:00, 16962.75 examples/s]

Map:  28%|███████████▉                               | 4000/14342 [00:00<00:00, 16712.79 examples/s]

Map:  42%|█████████████████▉                         | 6000/14342 [00:00<00:00, 16541.26 examples/s]

Map:  56%|███████████████████████▉                   | 8000/14342 [00:00<00:00, 16443.45 examples/s]

Map:  70%|█████████████████████████████▎            | 10000/14342 [00:00<00:00, 16438.61 examples/s]

Map:  84%|███████████████████████████████████▏      | 12000/14342 [00:00<00:00, 16443.35 examples/s]

Map:  98%|████████████████████████████████████████▉ | 14000/14342 [00:00<00:00, 16503.17 examples/s]

Map: 100%|██████████████████████████████████████████| 14342/14342 [00:00<00:00, 16474.05 examples/s]

In [22]:
for sample in preprocessed_test_dataset.select(range(5)):
    print(sample['input_ids'])
    print(sample['attention_mask'])
    print(sample['labels'])

[2, 2, 2, 1, 4103, 9632, 515, 831, 304, 321, 29877, 29901, 13, 267, 29901, 26692, 831, 1321, 8154, 2407, 279, 5409, 29889, 353, 321, 29877, 29901, 22388, 29871, 31431, 2829, 288, 637, 1540, 337, 1397, 29875, 980, 262, 29889, 13, 267, 29901, 1869, 7014, 316, 17176, 328, 343, 353, 321, 29877, 29901, 425, 1957, 3848, 3691, 4931, 423, 4303, 406, 1111, 413, 1175, 13, 267, 29901, 5516, 425, 3638, 29874, 316, 1232, 4439, 359, 282, 406, 23322, 29889, 353, 321, 29877, 29901, 7048, 425, 992, 2212, 316, 4439, 1631, 352, 3848, 282, 406, 359, 29889, 13, 267, 29901, 1354, 1750, 406, 21415, 20397, 17639, 353, 321, 29877, 29901, 409, 3516, 1700, 294, 2071, 1091, 29875, 1375, 13, 267, 29901, 831, 3079, 29983, 569, 2689, 6062, 4778, 29889, 353, 321, 29877, 29901, 22388, 452, 4109, 311, 569, 29871, 31303, 2386, 29889, 13, 267, 29901, 633, 307, 1232, 288, 14736, 29892, 553, 29880, 398, 1182, 912, 29889, 353, 321, 29877, 29901, 29871]
[0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

bitsandbytes is a quantization library with a Transformers integration. With this integration, you can quantize a model to 8 or 4-bits and enable many other options by configuring the BitsAndBytesConfig class. For example, you can:

<ul>
<li>set load_in_4bit=True to quantize the model to 4-bits when you load it</li>
<li>set bnb_4bit_quant_type="nf4" to use a special 4-bit data type for weights initialized from a normal distribution</li>
<li>set bnb_4bit_use_double_quant=True to use a nested quantization scheme to quantize the already quantized weights</li>
<li>set bnb_4bit_compute_dtype=torch.bfloat16 to use bfloat16 for faster computation</li>
</ul>


In [23]:
import torch
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

Pass the quantization_config to the from_pretrained method.

In [24]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    token=True,
    quantization_config=quantization_config,
    dtype=torch.bfloat16,
)


Loading checkpoint shards:   0%|                                              | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:  50%|███████████████████                   | 1/2 [00:03<00:03,  3.62s/it]

Loading checkpoint shards: 100%|██████████████████████████████████████| 2/2 [00:04<00:00,  2.25s/it]

Loading checkpoint shards: 100%|██████████████████████████████████████| 2/2 [00:04<00:00,  2.45s/it]

# Inference

Loading default inference parameters for the model, so that additional parameters could be added and passed to the [generate function](https://huggingface.co/docs/transformers/main_classes/text_generation):

In [25]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(
    checkpoint,
    )

print(generation_config)

GenerationConfig {
  "bos_token_id": 1,
  "do_sample": true,
  "eos_token_id": 2,
  "max_length": 4096,
  "pad_token_id": 0,
  "temperature": 0.6,
  "top_p": 0.9
}



As observed, the default search strategy for Llama-2 is Top-p with probability 0.9 and temperature 0.6 ($0<T<1$ amplifies output probability differences and makes output more deterministic). [The search strategy can be selected](https://huggingface.co/docs/transformers/en/generation_strategies) at inference time. 

First, the test set is divided in small batches to reduce GPU memory comsumption:

In [26]:
batch_tokenized_test = preprocessed_test_dataset.batch(CONFIG.batch_size)

Batching examples:   0%|                                           | 0/14342 [00:00<?, ? examples/s]

Batching examples:  10%|██▉                          | 1472/14342 [00:00<00:00, 14085.44 examples/s]

Batching examples:  21%|█████▉                       | 2944/14342 [00:00<00:00, 14214.51 examples/s]

Batching examples:  31%|████████▊                    | 4384/14342 [00:00<00:00, 14167.60 examples/s]

Batching examples:  41%|███████████▊                 | 5856/14342 [00:00<00:00, 14281.83 examples/s]

...

Batching examples:  61%|█████████████████▋           | 8768/14342 [00:00<00:00, 14249.89 examples/s]

Batching examples:  71%|███████████████████▉        | 10240/14342 [00:00<00:00, 14276.52 examples/s]

Batching examples:  85%|███████████████████████▉    | 12256/14342 [00:00<00:00, 13841.66 examples/s]

Batching examples:  96%|██████████████████████████▊ | 13728/14342 [00:00<00:00, 13956.39 examples/s]

Batching examples: 100%|████████████████████████████| 14342/14342 [00:01<00:00, 14017.72 examples/s]

In [27]:
import tqdm 

number_of_batches = len(batch_tokenized_test["input_ids"])
output_sequences = []
all_sources = []

for i in tqdm.tqdm(range(number_of_batches)):
    all_sources.append(batch_tokenized_test["input_ids"][i])
    with torch.no_grad():
        output_batch = model.generate(
            generation_config=generation_config, 
            input_ids=torch.tensor(batch_tokenized_test["input_ids"][i]).cuda(), 
            attention_mask=torch.tensor(batch_tokenized_test["attention_mask"][i]).cuda(), 
            max_length = max_tok_len, 
            num_beams=1, 
            do_sample=False,
        )
    output_sequences.extend(output_batch)


  0%|                                                                       | 0/449 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  0%|▏                                                              | 1/449 [00:07<57:40,  7.72s/it]

  0%|▎                                                              | 2/449 [00:15<57:17,  7.69s/it]

  1%|▍                                                              | 3/449 [00:22<56:36,  7.62s/it]

...

 99%|████████████████████████████████████████████████████████████▌| 446/449 [55:47<00:22,  7.50s/it]

100%|████████████████████████████████████████████████████████████▋| 447/449 [55:54<00:15,  7.50s/it]

100%|████████████████████████████████████████████████████████████▊| 448/449 [56:02<00:07,  7.50s/it]

100%|█████████████████████████████████████████████████████████████| 449/449 [56:06<00:00,  6.63s/it]

100%|█████████████████████████████████████████████████████████████| 449/449 [56:06<00:00,  7.50s/it]

## Evaluation

The output of the model is automatically evaluated compared to the reference translations. To this purpose, we use the [Evaluate library](https://huggingface.co/docs/evaluate) which includes the definition of generic and task-specific metrics. In our case, we use the [BLEU metric](https://huggingface.co/spaces/evaluate-metric/bleu), or to be more precise, [sacreBLEU](https://huggingface.co/spaces/evaluate-metric/sacrebleu).

In [28]:
from evaluate import load

metric_bleu = load("sacrebleu")
metric_comet = load("comet")

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Fetching 5 files:   0%|                                                       | 0/5 [00:00<?, ?it/s]

Fetching 5 files: 100%|███████████████████████████████████████████| 5/5 [00:00<00:00, 135300.13it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`


Encoder model frozen.


/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


The example below performs a basic post-processing to decode the predictions and extract the translation:

In [29]:
import re

def compute_metrics(sample, output_sequences):
    inputs = [f"{task_prefix}{shots}{src}: {s} = {tgt}: " for s in sample["source_text"]]
    preds = tokenizer.batch_decode(output_sequences, skip_special_tokens=True)
    # print(inputs)
    # print(preds)
    for i, (input,pred) in enumerate(zip(inputs,preds)):
      pred = re.search(r'^.*\n',pred.removeprefix(input).lstrip())
      if pred is not None:
        preds[i] = pred.group()[:-1]
      else:
        preds[i] = ""
    # print(sample["source_text"])
    # print(sample["dest_text"])
    # print(preds)
    result_bleu = metric_bleu.compute(
       predictions=preds, 
       references=sample["dest_text"]
    )
    result_comet = metric_comet.compute(
        sources=sample["source_text"],
        predictions=preds, 
        references=sample["dest_text"]
    )
    result = {
      "bleu": result_bleu["score"],
      "comet": result_comet["mean_score"]
      }
    return result

In [30]:
result = compute_metrics(preprocessed_test_dataset,output_sequences, )
print(f'BLEU score: {result["bleu"]:0.4f}')
print(f'COMET score: {result["comet"]:0.4f}')

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/torch/__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return _C._get_float32_matmul_precision()
You are using a CUDA device ('NVIDIA GeForce RTX 5070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/t

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


BLEU score: 4.6400
COMET score: 0.6062


In [31]:
from maikol_utils.print_utils import print_separator

# all_sources already contains raw text strings
# output_sequences contains token IDs that need to be decoded
decoded_outputs = tokenizer.batch_decode(output_sequences, skip_special_tokens=True)

# Get reference translations from test set
test_references = tokenized_datasets["test"]["dest_text"]

# Print first 10 examples
for i, (source, output, reference) in enumerate(zip(all_sources[:10], decoded_outputs[:10], test_references[:10])):
    print_separator(f"Example {i+1}:")
    print(f"Source:      {source}")
    print(f"Translation: {output}")
    print(f"Reference:   {reference}")

________________________________________________________________
                           Example 1:                           

Source:      [[2, 2, 2, 1, 4103, 9632, 515, 831, 304, 321, 29877, 29901, 13, 267, 29901, 26692, 831, 1321, 8154, 2407, 279, 5409, 29889, 353, 321, 29877, 29901, 22388, 29871, 31431, 2829, 288, 637, 1540, 337, 1397, 29875, 980, 262, 29889, 13, 267, 29901, 1869, 7014, 316, 17176, 328, 343, 353, 321, 29877, 29901, 425, 1957, 3848, 3691, 4931, 423, 4303, 406, 1111, 413, 1175, 13, 267, 29901, 5516, 425, 3638, 29874, 316, 1232, 4439, 359, 282, 406, 23322, 29889, 353, 321, 29877, 29901, 7048, 425, 992, 2212, 316, 4439, 1631, 352, 3848, 282, 406, 359, 29889, 13, 267, 29901, 1354, 1750, 406, 21415, 20397, 17639, 353, 321, 29877, 29901, 409, 3516, 1700, 294, 2071, 1091, 29875, 1375, 13, 267, 29901, 831, 3079, 29983, 569, 2689, 6062, 4778, 29889, 353, 321, 29877, 29901, 22388, 452, 4109, 311, 569, 29871, 31303, 2386, 29889, 13, 267, 29901, 633, 307, 1232, 288, 14736, 